# 08_track4_rfc: Track 4, linked changes and an asset watchlist
Objective 4 asks to predict RFCs and possible failure or misconfiguration of assets. The extract has two proxies: Has_Related_Change (whether a ticket is linked to a change, sections 1 to 9) and Closure_Code (what the cause turned out to be, section 10). Aggregates only: no change IDs, ticket IDs or CI names are printed. Needs `data/processed/itsm_clean.pkl`, xgboost and lightgbm; section 6 takes a few minutes.

**Contents**
- Sections 1 to 4: the target, how clustered the changes are, the five largest changes, a burst check.
- Sections 5 and 6: five classifiers with grid search, with and without the five largest changes.
- Section 7: the asset-token watchlist (the deliverable).
- Sections 8 and 9: late-linking check, inferred CI relationships.
- Section 10: likely failure type at creation (from Closure_Code).
- Section 11: findings and decision.

## 1. Setup, window and target profile

In [1]:
import sys
import time
from itertools import combinations

import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

sys.path.append("../src")
from features import set_b
from project_config import CUT, START, TEST_START

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

df = pd.read_pickle("../data/processed/itsm_clean.pkl")
w = df[(df["Open_Time"] >= START) & (df["Open_Time"] < CUT)].copy()
w["rfc"] = w["Has_Related_Change"].astype(int)

print("rows:", len(w), "| with a related change:", int(w["rfc"].sum()),
      "| rate %:", round(w["rfc"].mean() * 100, 2))
print("Table 1. Linked tickets per month (size = tickets, sum = linked, mean = linked rate)")
print(w.groupby(w["Open_Time"].dt.to_period("M"))["rfc"].agg(["size", "sum", "mean"]).round(3))
for t, c in zip([2, 3], ["CI_Cat", "Category"]):
    print(f"Table {t}. Linked rate by {c} (6 largest)")
    g = w.groupby(c)["rfc"].agg(rows="size", rfc="sum")
    g["rate_%"] = (g["rfc"] / g["rows"] * 100).round(2)
    print(g.sort_values("rows", ascending=False).head(6), "\n")

rows: 42304 | with a related change: 516 | rate %: 1.22
Table 1. Linked tickets per month (size = tickets, sum = linked, mean = linked rate)
           size  sum   mean
Open_Time                  
2013-10    8606   74  0.009
2013-11    8128  171  0.021
2013-12    6743   85  0.013
2014-01    8270   47  0.006
2014-02    7377  126  0.017
2014-03    3180   13  0.004
Table 2. Linked rate by CI_Cat (6 largest)
                 rows  rfc  rate_%
CI_Cat                            
application     29798  369    1.24
subapplication   7060  101    1.43
computer         3377   24    0.71
storage           635    0    0.00
hardware          399    1    0.25
software          303   15    4.95 

Table 3. Linked rate by Category (6 largest)
                          rows  rfc  rate_%
Category                                   
incident                 34134  489    1.43
request for information   8159   27    0.33
complaint                   10    0    0.00
request for change           1    0    0.00 


<details>
<summary><b>Code notes, line by line</b></summary>

- `import ... combinations` : Pairs of items (section 9).
- `import lightgbm as lgb` : Gradient boosting library used by the asset-token model.
- `from scipy import sparse` : Sparse matrices for the CI-by-hour table (section 9).
- `from sklearn ... / XGBClassifier` : Encoders, models, search and metrics.
- `from features import set_b; from project_config import ...` : Shared features and dates.
- `w = df[(Open_Time >= START) & (Open_Time < CUT)].copy()` : The modelling window.
- `w["rfc"] = w["Has_Related_Change"].astype(int)` : Target: 1 if a change is linked.
- `print("rows:", ..., "rate %:", ...)` : Size and rate of the target.
- `groupby(month)["rfc"].agg(["size", "sum", "mean"])` : Linked tickets per month.
- `for c in ["CI_Cat", "Category"]: ...` : Linked rate by asset type and category (6 largest).

</details>

**Result.** 516 of 42,304 window tickets (1.22%) have a linked change. The monthly rate swings from 0.4% (1 to 16 March) to 2.1% (November) (Table 1). Software tickets are linked most often (4.95%) and storage never (Table 2); incidents at 1.43% against 0.33% for requests for information (Table 3).

### 1b. Resolution time: linked against not linked

In [2]:
w["Elapsed_hrs"] = (w["Close_Time"] - w["Open_Time"]).dt.total_seconds() / 3600
summary = w.groupby("rfc")["Elapsed_hrs"].agg(
    tickets="size", median_hrs="median",
    closed_within_1h=lambda s: round((s <= 1).mean() * 100, 1),
    open_over_7_days=lambda s: round((s > 168).mean() * 100, 1))
summary.index = ["Not linked", "Linked to a change"]
print("Table 4. Resolution time: linked against not linked")
display(summary)

Table 4. Resolution time: linked against not linked


,tickets,median_hrs,closed_within_1h,open_over_7_days
Not linked,41788,18.183333,22.1,15.0
Linked to a change,516,6.350000,19.6,14.1


<details>
<summary><b>Code notes, line by line</b></summary>

- `w["Elapsed_hrs"] = ...` : Hours from open to close.
- `groupby("rfc")["Elapsed_hrs"].agg(tickets, median_hrs, closed_within_1h, open_over_7_days)` : Linked against not linked: count, median hours, % closed within 1 hour, % open over 7 days (168 hours).

</details>

**Result.** Linked tickets close faster in the middle of the range (median 6.4 hours against 18.2), but the shares closed within an hour (19.6% against 22.1%) and open longer than 7 days (14.1% against 15.0%) are similar (Table 4).

### 1c. No_of_Related_Changes against the target

In [3]:
print("Table 5. No_of_Related_Changes against the target")
print(pd.crosstab(w["rfc"], w["No_of_Related_Changes"].fillna(-1).astype(int),
                  rownames=["Has_Related_Change"], colnames=["No_of_Related_Changes (-1 = missing)"]))

Table 5. No_of_Related_Changes against the target
No_of_Related_Changes (-1 = missing)      0    1   2  3  9
Has_Related_Change                                        
0                                     41788    0   0  0  0
1                                         0  498  16  1  1


<details>
<summary><b>Code notes, line by line</b></summary>

- `pd.crosstab(w["rfc"], w["No_of_Related_Changes"].fillna(-1).astype(int), ...)` : Target against the change count (-1 = missing): if they match perfectly, the count is the target in disguise (leakage).

</details>

**Result.** No_of_Related_Changes equals the target in another form (Table 5): 0 for all 41,788 unlinked tickets and 1 or more for all 516 linked ones (498 have 1, 16 have 2, 1 has 3, 1 has 9). Excluded as a feature (leakage) and not used as a separate target.

## 2. How clustered are the linked changes?

In [4]:
pos = w[w["rfc"] == 1]
per_change = pos["Related_Change"].astype(str).value_counts()

print("Table 6. How clustered the linked changes are")
print("positive tickets:", len(pos), "| distinct changes:", len(per_change))
print("tickets per change:", per_change.describe().round(1).to_dict())
print("share of positives on the 5 largest changes:",
      round(per_change.head(5).sum() / len(pos) * 100, 1), "% | largest single change:", int(per_change.iloc[0]), "tickets")
print(pos.groupby(pos["Open_Time"].dt.to_period("M")).agg(tickets=("rfc", "size"), changes=("Related_Change", "nunique")))
print("distinct CI_Name with a change:", pos["CI_Name"].nunique(), "of", w["CI_Name"].nunique())


Table 6. How clustered the linked changes are
positive tickets: 516 | distinct changes: 205
tickets per change: {'count': 205.0, 'mean': 2.5, 'std': 9.5, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 1.0, 'max': 110.0}
share of positives on the 5 largest changes: 45.0 % | largest single change: 110 tickets
           tickets  changes
Open_Time                  
2013-10         74       49
2013-11        171       42
2013-12         85       44
2014-01         47       31
2014-02        126       38
2014-03         13       12
distinct CI_Name with a change: 132 of 2861


<details>
<summary><b>Code notes, line by line</b></summary>

- `pos = w[w["rfc"] == 1]` : Linked tickets.
- `per_change = pos["Related_Change"].astype(str).value_counts()` : Tickets per change (IDs are used for counting, never printed).
- `per_change.describe()` : Distribution of tickets per change.
- `per_change.head(5).sum() / len(pos)` : Share of linked tickets on the 5 largest changes.
- `pos.groupby(month).agg(tickets=..., changes=nunique)` : Linked tickets and distinct changes per month.
- `pos["CI_Name"].nunique()` : How many assets ever had a linked ticket.

</details>

**Result.** The 516 linked tickets belong to 205 changes: the median change links 1 ticket, the largest 110, and the 5 largest hold 45.0% of linked tickets (Table 6). The busy months (November 171 linked tickets, February 126) have no more distinct changes than the others (42 and 38): a few large changes drive the monthly swings in Table 1. Only 132 of 2,861 CIs ever had a linked ticket.

## 3. The largest changes and the train/test overlap

In [5]:
top5 = per_change.head(5).index
print("Table 7. The five largest changes (by size rank) and the train/test overlap")
for rank, c in enumerate(top5, 1):
    t = pos[pos["Related_Change"].astype(str) == c]
    print(f"change #{rank}: {len(t)} tickets | opened {t['Open_Time'].min().date()} to {t['Open_Time'].max().date()} | "
          f"{t['CI_Cat'].nunique()} CI_Cat, {t['CI_Name'].nunique()} CI_Name, {t['WBS'].nunique()} WBS | "
          f"main CI_Cat: {t['CI_Cat'].value_counts().index[0]}")

train_ids = set(pos.loc[pos["Open_Time"] < TEST_START, "Related_Change"].astype(str))
test_pos = pos[pos["Open_Time"] >= TEST_START]
test_ids = test_pos["Related_Change"].astype(str)
print("train positives:", int((pos["Open_Time"] < TEST_START).sum()),
      "| test positives:", len(test_pos), "| distinct changes in test:", test_ids.nunique())
print("test positives whose change also has train tickets:", int(test_ids.isin(train_ids).sum()))

Table 7. The five largest changes (by size rank) and the train/test overlap
change #1: 110 tickets | opened 2013-11-04 to 2013-11-08 | 2 CI_Cat, 6 CI_Name, 1 WBS | main CI_Cat: application
change #2: 78 tickets | opened 2014-02-17 to 2014-02-27 | 2 CI_Cat, 4 CI_Name, 2 WBS | main CI_Cat: application
change #3: 18 tickets | opened 2013-10-07 to 2014-03-10 | 2 CI_Cat, 12 CI_Name, 11 WBS | main CI_Cat: application
change #4: 16 tickets | opened 2013-12-02 to 2013-12-02 | 2 CI_Cat, 4 CI_Name, 1 WBS | main CI_Cat: subapplication
change #5: 10 tickets | opened 2013-11-14 to 2013-11-18 | 1 CI_Cat, 1 CI_Name, 1 WBS | main CI_Cat: subapplication
train positives: 377 | test positives: 139 | distinct changes in test: 49
test positives whose change also has train tickets: 6


<details>
<summary><b>Code notes, line by line</b></summary>

- `top5 = per_change.head(5).index` : The five largest changes.
- `for rank, c in enumerate(top5, 1): ...` : For each, by rank only: tickets, date range, number of distinct asset types, CIs and WBS, and its main asset type.
- `train_ids = set(...); test_ids = ...` : Changes seen in training and in the test period.
- `test_ids.isin(train_ids).sum()` : Test linked tickets whose change already had tickets in training (shared storms).

</details>

### 3b. Profile of the five largest changes

In [6]:
top5_tickets = pos[pos["Related_Change"].astype(str).isin(top5)].copy()
top5_tickets["Change"] = top5_tickets["Related_Change"].astype(str).map({c: f"#{i}" for i, c in enumerate(top5, 1)})
top5_tickets["Priority label"] = top5_tickets["Priority"].map(lambda p: f"P{int(p)}" if pd.notna(p) else "unknown")

by_category = pd.crosstab(top5_tickets["Change"], top5_tickets["Category"], margins=True, margins_name="Total")
by_category.columns.name = None

by_priority = pd.crosstab(top5_tickets["Change"], top5_tickets["Priority label"], margins=True, margins_name="Total")
by_priority.columns.name = None

timing = top5_tickets.groupby("Change").agg(
    Tickets=("Open_Time", "size"),
    First_opened=("Open_Time", lambda s: s.min().date()),
    Last_opened=("Open_Time", lambda s: s.max().date()),
    Share_on_weekdays=("Open_Time", lambda s: round(s.dt.day_of_week.lt(5).mean(), 2)),
)

print("Table 8. Tickets per change, by category (all are incidents: break-fix)")
display(by_category)
print("Table 9. Tickets per change, by priority")
display(by_priority)
print("Table 10. Size, date range and weekday share per change")
display(timing)

Table 8. Tickets per change, by category (all are incidents: break-fix)


,incident,Total
Change,,
#1,110,110
#2,78,78
#3,18,18
#4,16,16
#5,10,10
Total,232,232


Table 9. Tickets per change, by priority


,P2,P3,P4,P5,unknown,Total
Change,,,,,,
#1,0,0,110,0,0,110
#2,0,38,23,2,15,78
#3,2,2,5,9,0,18
#4,0,1,14,0,1,16
#5,0,0,1,9,0,10
Total,2,41,153,20,16,232


Table 10. Size, date range and weekday share per change


,Tickets,First_opened,Last_opened,Share_on_weekdays
Change,,,,
#1,110,2013-11-04,2013-11-08,1.00
#2,78,2014-02-17,2014-02-27,1.00
#3,18,2013-10-07,2014-03-10,0.94
#4,16,2013-12-02,2013-12-02,1.00
#5,10,2013-11-14,2013-11-18,1.00


<details>
<summary><b>Code notes, line by line</b></summary>

- `top5_tickets["Change"] = ...map({c: f"#{i}" ...})` : Replaces each change ID with a rank label (#1 to #5).
- `top5_tickets["Priority label"] = ...` : P1 to P5, or 'unknown'.
- `pd.crosstab(..., margins=True, margins_name="Total")` : Tables 8 and 9: tickets per change by category and by priority, with totals.
- `timing = groupby("Change").agg(Tickets, First_opened, Last_opened, Share_on_weekdays)` : Table 10: size, dates and weekday share per change.

</details>

**Result.** The five largest changes are short bursts: #1 links 110 tickets in 5 days (4 to 8 November), #2 78 tickets in 11 days and #4 16 tickets on one day; only #3 is spread over five months (Tables 7 and 10). All 232 are incidents (Table 8), mostly P4 (153 of 232) with 2 P2 (Table 9), and almost all were opened on weekdays (Table 10). Only 6 of the 139 test positives belong to a change that also has training tickets (Table 7), so the test period is mostly new changes.

## 4. Does a burst on the same WBS or CI predict a linked change?

In [7]:
w = w.sort_values("Open_Time").reset_index(drop=True)
tt = w["Open_Time"].to_numpy()

def prior_count(key, hours):
    """Tickets on the same key opened in the previous `hours`, before this ticket."""
    out = np.zeros(len(w))
    for idx in w.groupby(key).indices.values():
        t = tt[idx]
        left = np.searchsorted(t, t - np.timedelta64(hours, "h"), side="left")
        out[idx] = np.arange(len(idx)) - left
    return out

w["wbs_24h"] = prior_count("WBS", 24)
w["ci_24h"] = prior_count("CI_Name", 24)
w["in_top5"] = w["Related_Change"].astype(str).isin(top5)
w["rfc_ex"] = ((w["rfc"] == 1) & ~w["in_top5"]).astype(int)

bins = [-1, 0, 2, 9, 49, 1e9]
for col in ["wbs_24h", "ci_24h"]:
    g = w.groupby(pd.cut(w[col], bins), observed=True).agg(
        tickets=("rfc", "size"), linked=("rfc", "sum"),
        rate=("rfc", "mean"), rate_without_top5=("rfc_ex", "mean"))
    print(f"Table {11 if col == 'wbs_24h' else 12}. Linked rate (%) by tickets on the same {'WBS' if col == 'wbs_24h' else 'CI'} in the previous 24 hours")
    print((g * [1, 1, 100, 100]).round(2), "\n")

Table 11. Linked rate (%) by tickets on the same WBS in the previous 24 hours
                      tickets  linked  rate  rate_without_top5
wbs_24h                                                       
(-1.0, 0.0]              4754     100  2.10               1.89
(0.0, 2.0]               5808      69  1.19               1.08
(2.0, 9.0]              11144      79  0.71               0.61
(9.0, 49.0]              9841     110  1.12               0.25
(49.0, 1000000000.0]    10757     158  1.47               0.35 

Table 12. Linked rate (%) by tickets on the same CI in the previous 24 hours
                      tickets  linked  rate  rate_without_top5
ci_24h                                                        
(-1.0, 0.0]             10892     144  1.32               1.16
(0.0, 2.0]               8466      99  1.17               0.93
(2.0, 9.0]              12844     100  0.78               0.45
(9.0, 49.0]             10088     173  1.71               0.21
(49.0, 1000000000.0]    

<details>
<summary><b>Code notes, line by line</b></summary>

- `w = w.sort_values("Open_Time").reset_index(drop=True); tt = ...` : Time order and the opening times as an array.
- `def prior_count(key, hours):` : Earlier tickets on the same key within the past `hours` (searchsorted finds the window start).
- `w["wbs_24h"], w["ci_24h"]` : Burst features: tickets on the same WBS and the same CI in the previous 24 hours.
- `w["in_top5"]; w["rfc_ex"]` : Whether a ticket belongs to a top-5 change, and the target without them.
- `bins = [-1, 0, 2, 9, 49, 1e9]` : Groups: 0, 1-2, 3-9, 10-49, 50+ earlier tickets.
- `groupby(pd.cut(w[col], bins)).agg(tickets, linked, rate, rate_without_top5)` : Linked rate by burst size, with and without the storms.
- `(g * [1, 1, 100, 100]).round(2)` : Rates shown as %.

</details>

**Result.** A burst does not signal a change. Without the five largest changes, the linked rate falls as the number of recent tickets on the same WBS rises, from 1.89% (none in the previous 24 hours) to 0.25% and 0.35% (10 or more) (Table 11, last column); on the same CI it falls from 1.16% to 0.21% (Table 12). The raw rates rise again at 10 or more only because the five storms sit there.

## 5. Modelling setup: features and time split

In [8]:
w["hour"] = w["Open_Time"].dt.hour
w["weekday"] = w["Open_Time"].dt.day_of_week
tr, te = w[w["Open_Time"] < TEST_START], w[w["Open_Time"] >= TEST_START]
print("train:", len(tr), int(tr["rfc"].sum()), "| test:", len(te), int(te["rfc"].sum()))

train: 31747 377 | test: 10557 139


<details>
<summary><b>Code notes, line by line</b></summary>

- `w["hour"], w["weekday"]` : Time features.
- `tr, te = w[< TEST_START], w[>= TEST_START]` : Same split as the other tracks.

</details>

## 6. All five models with grid search
Two setups: all tickets, and without the five largest changes. Precision, recall and F1 use each model's default 0.5 cut-off; average precision, ROC-AUC and the top-2.2% count use the scores.

In [9]:
models = {
    "Logistic Regression": (LogisticRegression(max_iter=2000),
        {"clf__C": [0.01, 0.1, 1, 10]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=42),
        {"clf__max_depth": [3, 5, 8], "clf__min_samples_leaf": [20, 50]}),
    "Random Forest": (RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
        {"clf__max_depth": [6, 10], "clf__min_samples_leaf": [5, 20]}),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=42),
        {"clf__n_estimators": [100, 200], "clf__learning_rate": [0.05, 0.1], "clf__max_depth": [2, 3]}),
    "XGBoost": (XGBClassifier(n_estimators=200, eval_metric="logloss", random_state=42, n_jobs=-1),
        {"clf__max_depth": [3, 5], "clf__learning_rate": [0.05, 0.1], "clf__min_child_weight": [1, 5]}),
}

def make_pipe(cols, clf):
    enc = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30)
    return Pipeline([("pre", ColumnTransformer([("cat", enc, cols)])), ("clf", clf)])

def tune(train):
    """Grid search each algorithm with time-ordered CV on the training months."""
    train = train.sort_values("Open_Time")
    X, y = train[set_b].astype(str), train["rfc"]
    cv = TimeSeriesSplit(n_splits=3)
    wts = compute_sample_weight("balanced", y)
    fitted = {}
    for name, (clf, grid) in models.items():
        gs = GridSearchCV(make_pipe(set_b, clf), grid, cv=cv, scoring="average_precision", n_jobs=1)
        gs.fit(X, y, clf__sample_weight=wts)
        fitted[name] = gs
    fold_positives = [int(y.iloc[v].sum()) for _, v in cv.split(X)]
    return fitted, int(y.sum()), fold_positives

def evaluate(fitted, train, test):
    """One row per algorithm; also the classification report of the best model by CV average precision."""
    Xtr, Xte = train[set_b].astype(str), test[set_b].astype(str)
    ytr, yte = train["rfc"], test["rfc"]
    rows = []
    for name, gs in fitted.items():
        m = gs.best_estimator_
        p_tr, p_te = m.predict(Xtr), m.predict(Xte)
        proba = m.predict_proba(Xte)[:, 1]
        k = int(np.ceil(0.022 * len(test)))          # alert on the top 2.2% of tickets, as in Track 1
        rows.append({"Model": name,
                     "CV AP (train)": gs.best_score_,
                     "Train accuracy": accuracy_score(ytr, p_tr),
                     "Test accuracy": accuracy_score(yte, p_te),
                     "Recall": recall_score(yte, p_te),
                     "Precision": precision_score(yte, p_te, zero_division=0),
                     "F1": f1_score(yte, p_te),
                     "ROC-AUC": roc_auc_score(yte, proba),
                     "Avg precision": average_precision_score(yte, proba),
                     "AP / base rate": average_precision_score(yte, proba) / yte.mean(),
                     "Linked in top 2.2%": int(yte.to_numpy()[np.argsort(-proba)[:k]].sum())})
    best = max(fitted, key=lambda k: fitted[k].best_score_)
    report = classification_report(yte, fitted[best].best_estimator_.predict(Xte),
                                   target_names=["No change", "Linked change"], digits=3, zero_division=0)
    return pd.DataFrame(rows).set_index("Model").round(3), best, report

setups = [("ALL TICKETS", tr, te),
          ("WITHOUT THE 5 LARGEST CHANGES", tr[~tr["in_top5"]], te[~te["in_top5"]])]
results_t4, settings_t4 = {}, {}
for t, (label, a, b) in zip([13, 14], setups):
    fitted, n_pos, fold_pos = tune(a)
    res, best, report = evaluate(fitted, a, b)
    results_t4[label] = res
    settings_t4[label] = pd.Series({k: str(g.best_params_) for k, g in fitted.items()})
    print("=" * 12, f"Table {t}. {label}", "=" * 12)
    print(f"train positives {n_pos} (per validation fold {fold_pos}) | test positives {int(b['rfc'].sum())} | "
          f"base rate {b['rfc'].mean():.4f} | always-'no change' accuracy {1 - b['rfc'].mean():.3f}")
    display(res)
    print(f"Classification report, best model by CV average precision: {best}")
    print(report)

print("Table 15. Best grid-search settings")
display(pd.DataFrame(settings_t4))

============ Table 13. ALL TICKETS ============
train positives 377 (per validation fold [172, 90, 45]) | test positives 139 | base rate 0.0132 | always-'no change' accuracy 0.987


,CV AP (train),Train accuracy,Test accuracy,Recall,Precision,F1,ROC-AUC,Avg precision,AP / base rate,Linked in top 2.2%
Model,,,,,,,,,,
Logistic Regression,0.026,0.697,0.645,0.324,0.012,0.023,0.443,0.016,1.201,7
Decision Tree,0.026,0.396,0.344,0.597,0.012,0.023,0.447,0.015,1.147,5
Random Forest,0.019,0.695,0.637,0.302,0.011,0.021,0.415,0.013,1.013,5
Gradient Boosting,0.022,0.663,0.593,0.317,0.010,0.020,0.382,0.018,1.361,11
XGBoost,0.024,0.780,0.728,0.252,0.013,0.024,0.443,0.015,1.125,9


Classification report, best model by CV average precision: Decision Tree
               precision    recall  f1-score   support

    No change      0.984     0.340     0.506     10418
Linked change      0.012     0.597     0.023       139

     accuracy                          0.344     10557
    macro avg      0.498     0.469     0.265     10557
 weighted avg      0.972     0.344     0.499     10557

============ Table 14. WITHOUT THE 5 LARGEST CHANGES ============
train positives 228 (per validation fold [54, 69, 40]) | test positives 56 | base rate 0.0053 | always-'no change' accuracy 0.995


,CV AP (train),Train accuracy,Test accuracy,Recall,Precision,F1,ROC-AUC,Avg precision,AP / base rate,Linked in top 2.2%
Model,,,,,,,,,,
Logistic Regression,0.024,0.820,0.811,0.554,0.016,0.030,0.739,0.026,4.916,9
Decision Tree,0.018,0.930,0.927,0.375,0.028,0.052,0.649,0.024,4.413,10
Random Forest,0.028,0.764,0.755,0.536,0.012,0.023,0.749,0.023,4.308,10
Gradient Boosting,0.030,0.835,0.839,0.500,0.017,0.032,0.735,0.025,4.632,12
XGBoost,0.037,0.816,0.810,0.518,0.015,0.028,0.724,0.034,6.422,14


Classification report, best model by CV average precision: XGBoost
               precision    recall  f1-score   support

    No change      0.997     0.811     0.895     10418
Linked change      0.015     0.518     0.028        56

     accuracy                          0.810     10474
    macro avg      0.506     0.665     0.461     10474
 weighted avg      0.992     0.810     0.890     10474

Table 15. Best grid-search settings


,ALL TICKETS,WITHOUT THE 5 LARGEST CHANGES
Logistic Regression,{'clf__C': 10},{'clf__C': 0.01}
Decision Tree,"{'clf__max_depth': 8, 'clf__min_samples_leaf':...","{'clf__max_depth': 8, 'clf__min_samples_leaf':..."
Random Forest,"{'clf__max_depth': 10, 'clf__min_samples_leaf'...","{'clf__max_depth': 6, 'clf__min_samples_leaf':..."
Gradient Boosting,"{'clf__learning_rate': 0.1, 'clf__max_depth': ...","{'clf__learning_rate': 0.1, 'clf__max_depth': ..."
XGBoost,"{'clf__learning_rate': 0.1, 'clf__max_depth': ...","{'clf__learning_rate': 0.05, 'clf__max_depth':..."


<details>
<summary><b>Code notes, line by line</b></summary>

- `models = {...}` : The five algorithms with small grids.
- `def make_pipe(cols, clf):` : One-hot encoder (rare values pooled) + model.
- `def tune(train):` : Grid search per algorithm: time-ordered 3-fold CV, average precision, balanced weights.
- `fold_positives = [int(y.iloc[v].sum()) for _, v in cv.split(X)]` : Linked tickets in each validation fold, to show how few there are.
- `def evaluate(fitted, train, test):` : Scores at 0.5, ROC-AUC, AP, AP / base rate, and linked tickets in the top 2.2%.
- `k = int(np.ceil(0.022 * len(test)))` : Alert budget of 2.2% of tickets, as in Track 1.
- `setups = [("ALL TICKETS", tr, te), ("WITHOUT THE 5 LARGEST CHANGES", ...)]` : Two setups: does a signal remain without the storms?
- `for label, a, b in setups: ...` : Tunes, scores and prints the table and report for each setup.
- `display(pd.DataFrame(settings_t4))` : Best settings per algorithm and setup.

</details>

**Result.** On all tickets every model is at or below chance: ROC-AUC 0.38 to 0.45, and average precision 1.0 to 1.4 times the 0.013 base rate (Table 13). Without the five largest changes, ROC-AUC rises to 0.65 to 0.75 and average precision to 4.3 to 6.4 times the 0.0053 base rate, but precision stays at 1.2% to 2.8% (Table 14). In both setups test accuracy (0.34 to 0.93) is below always predicting "no change" (0.987 and 0.995). Table 15 lists the chosen settings.

## 7. Asset-token model: a short watchlist

TF-IDF, a text-vectorisation technique from NLP and information retrieval, is used here only as an encoder for the ticket's asset codes (CI_Name, CI_Cat, CI_Subcat, WBS, Category); the extract has no free text.

### 7a. Trained on earlier months, scored on the test period

In [10]:
text_cols = ["CI_Name", "CI_Cat", "CI_Subcat", "WBS", "Category"]
w["asset_text"] = w[text_cols].fillna("").astype(str).agg(" ".join, axis=1)
tr, te = w[w["Open_Time"] < TEST_START], w[w["Open_Time"] >= TEST_START]

def token_model(ngram=(1, 1)):
    """TF-IDF over the ticket's asset codes (every code kept, rare CIs included), then LightGBM."""
    return Pipeline([("tfidf", TfidfVectorizer(max_features=5000, ngram_range=ngram)),
                     ("clf", lgb.LGBMClassifier(objective="binary", class_weight="balanced",
                                                random_state=42, n_jobs=-1, verbose=-1))])

asset_model = token_model().fit(tr["asset_text"], tr["rfc"])
p_asset = asset_model.predict_proba(te["asset_text"])[:, 1]
y_asset = te["rfc"].to_numpy()
order_te = np.argsort(-p_asset)

t12 = pd.DataFrame({"Value": [f"{y_asset.mean():.4f}",
                              f"{roc_auc_score(y_asset, p_asset):.3f}",
                              f"{average_precision_score(y_asset, p_asset):.3f}"]
                             + [str(int(y_asset[order_te[:k]].sum())) for k in [25, 50, 100, 233]]},
                   index=["Base rate", "ROC-AUC", "Average precision"]
                         + [f"Linked tickets in the top {k}" for k in [25, 50, 100, 233]])
print("Table 16. Asset-token model, test period (1 Feb to 16 Mar)")
display(t12)

Table 16. Asset-token model, test period (1 Feb to 16 Mar)


,Value
Base rate,0.0132
ROC-AUC,0.457
Average precision,0.064
Linked tickets in the top 25,6
Linked tickets in the top 50,14
Linked tickets in the top 100,21
Linked tickets in the top 233,25


<details>
<summary><b>Code notes, line by line</b></summary>

- `text_cols = [...]; w["asset_text"] = ...agg(" ".join, axis=1)` : Joins each ticket's asset codes into one short 'sentence'.
- `def token_model(ngram=(1, 1)):` : TF-IDF on those codes, then LightGBM.
- `TfidfVectorizer(max_features=5000, ngram_range=ngram)` : One weight per code (up to 5,000); rare codes are kept, unlike the one-hot encoder's pooling.
- `lgb.LGBMClassifier(objective="binary", class_weight="balanced", ...)` : Boosted trees with the rare class up-weighted.
- `asset_model = token_model().fit(tr["asset_text"], tr["rfc"])` : Trained on earlier months.
- `order_te = np.argsort(-p_asset)` : Test tickets ranked by score.
- `t12 = ...` : Table 16: base rate, ROC-AUC, AP and linked tickets in the top 25, 50, 100 and 233.

</details>

**Result.** Over the whole test period the asset-token model ranks poorly (ROC-AUC 0.457), but the top of its list is rich: 14 linked tickets in the top 50 and 21 in the top 100, against about 0.7 and 1.3 expected by chance at the 1.32% base rate; average precision 0.064, about 5 times the base rate (Table 16).

### 7b. Are the top hits one storm or many changes?

In [11]:
size = w.loc[w["rfc"] == 1, "Related_Change"].astype(str).value_counts()
rank = {c: i for i, c in enumerate(size.index, 1)}          # 1 = largest change
hits = te.iloc[order_te[:100]]
hits = hits[hits["rfc"] == 1]
t13 = (hits["Related_Change"].astype(str).map(rank).value_counts().sort_index()
          .rename("Linked tickets").rename_axis("Size rank of their change (1 = largest)").to_frame())
print("Table 17. Linked tickets in the top 100, by size rank of their change (no change IDs shown)")
display(t13)
print("Distinct changes among them:", hits["Related_Change"].nunique(),
      "| share whose CI_Name also appears in training:", round(hits["CI_Name"].isin(tr["CI_Name"]).mean(), 2))

Table 17. Linked tickets in the top 100, by size rank of their change (no change IDs shown)


,Linked tickets
Size rank of their change (1 = largest),
3,1
21,3
44,2
46,2
165,1
169,1
173,1
175,1
179,1


Distinct changes among them: 17 | share whose CI_Name also appears in training: 1.0


<details>
<summary><b>Code notes, line by line</b></summary>

- `size = ...value_counts(); rank = {c: i ...}` : Size rank of every change (1 = largest).
- `hits = te.iloc[order_te[:100]]; hits[hits["rfc"] == 1]` : Linked tickets in the top 100.
- `...map(rank).value_counts().sort_index()` : Table 17: how many hits belong to each change rank (no IDs shown).
- `hits["Related_Change"].nunique(); hits["CI_Name"].isin(tr["CI_Name"]).mean()` : Distinct changes among the hits, and the share on CIs already seen in training.

</details>

**Result.** The 21 linked tickets in the top 100 come from 17 different changes, mostly small ones (size ranks 21 to 199; only one ticket from a top-5 change, rank 3), and all sit on CIs already seen in training (Table 17): the model flags recurring assets, not a single storm.

### 7c. Rolling check, and comparison with one-hot + CI_Name

Each period is scored by a model trained only on the months before it.

In [12]:
folds = [("2013-12-01", "2014-01-01", "Dec 2013"),
         ("2014-01-01", "2014-02-01", "Jan 2014"),
         ("2014-02-01", "2014-03-17", "Feb to 16 Mar 2014")]
periods = [f[2] for f in folds]
cols_ci = set_b + ["CI_Name"]

def onehot_ci(clf):
    enc = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=5)
    return Pipeline([("pre", ColumnTransformer([("cat", enc, cols_ci)])), ("clf", clf)])

candidates = {
    "Asset tokens, LightGBM": ("text", lambda: token_model()),
    "One-hot + CI_Name, XGBoost": ("onehot", lambda: onehot_ci(XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=5, min_child_weight=5,
        eval_metric="logloss", random_state=42, n_jobs=-1))),
    "One-hot + CI_Name, Logistic Regression": ("onehot", lambda: onehot_ci(
        LogisticRegression(C=0.01, max_iter=2000))),
}

rows = []
for a, b, label in folds:
    trf, tef = w[w["Open_Time"] < a], w[(w["Open_Time"] >= a) & (w["Open_Time"] < b)]
    y = tef["rfc"].to_numpy()
    for name, (kind, make) in candidates.items():
        m = make()
        if kind == "text":
            m.fit(trf["asset_text"], trf["rfc"])
            p = m.predict_proba(tef["asset_text"])[:, 1]
        else:
            m.fit(trf[cols_ci].astype(str), trf["rfc"],
                  clf__sample_weight=compute_sample_weight("balanced", trf["rfc"]))
            p = m.predict_proba(tef[cols_ci].astype(str))[:, 1]
        rows.append({"Model": name, "Period": label,
                     "Linked in top 50": int(y[np.argsort(-p)[:50]].sum()),
                     "AP": average_precision_score(y, p)})
    rows.append({"Model": "Chance (expected)", "Period": label,
                 "Linked in top 50": round(50 * y.mean(), 1), "AP": y.mean()})

res7 = pd.DataFrame(rows)
t14 = res7.pivot(index="Model", columns="Period", values="Linked in top 50")[periods]
t14["Total"] = t14.sum(axis=1)
t15 = res7.pivot(index="Model", columns="Period", values="AP")[periods].round(3)
t15["Mean"] = t15.mean(axis=1).round(3)
print("Table 18. Linked tickets among the 50 highest-scored tickets, per period")
display(t14)
print("Table 19. Average precision per period")
display(t15)

Table 18. Linked tickets among the 50 highest-scored tickets, per period


Period,Dec 2013,Jan 2014,Feb to 16 Mar 2014,Total
Model,,,,
"Asset tokens, LightGBM",10.0,8.0,14.0,32.0
Chance (expected),0.6,0.3,0.7,1.6
"One-hot + CI_Name, Logistic Regression",0.0,0.0,4.0,4.0
"One-hot + CI_Name, XGBoost",0.0,5.0,10.0,15.0


Table 19. Average precision per period


Period,Dec 2013,Jan 2014,Feb to 16 Mar 2014,Mean
Model,,,,
"Asset tokens, LightGBM",0.080,0.114,0.064,0.086
Chance (expected),0.013,0.006,0.013,0.011
"One-hot + CI_Name, Logistic Regression",0.029,0.009,0.019,0.019
"One-hot + CI_Name, XGBoost",0.037,0.093,0.086,0.072


<details>
<summary><b>Code notes, line by line</b></summary>

- `folds = [...]` : Three test periods, each trained on the months before.
- `cols_ci = set_b + ["CI_Name"]` : Set B plus the individual asset.
- `def onehot_ci(clf):` : One-hot pipeline with min_frequency=5 (so more individual CIs keep their own column).
- `candidates = {...}` : Asset tokens + LightGBM, one-hot + CI_Name with XGBoost, and with logistic regression.
- `if kind == "text": fit on asset_text else: fit on cols_ci with balanced weights` : Each candidate is trained in its own format.
- `"Linked in top 50": y[np.argsort(-p)[:50]].sum()` : Hits in a 50-ticket watchlist.
- `"Chance (expected)": 50 * y.mean()` : What a random list of 50 would contain.
- `t14 = pivot(...); t14["Total"]` : Table 18: hits per period and in total.
- `t15 = pivot(... "AP"); t15["Mean"]` : Table 19: AP per period and mean.

</details>

**Result.** Trained only on earlier months, the asset-token model puts 10, 8 and 14 linked tickets in its top 50 in December, January and February to March: 32 in total against 1.6 expected by chance, and ahead of one-hot + CI_Name with XGBoost (15) or logistic regression (4) (Table 18). Its mean average precision is 0.086, against 0.072 and 0.019 for the one-hot models and about 0.011 for chance (Table 19).

### 7d. Which part of the asset-token model matters?

In [13]:
variants = {
    "All fields (chosen)": (text_cols, (1, 1)),
    "All fields + pairs of neighbouring fields": (text_cols, (1, 2)),
    "Without CI_Name": ([c for c in text_cols if c != "CI_Name"], (1, 1)),
    "Without WBS": ([c for c in text_cols if c != "WBS"], (1, 1)),
}
rows = []
for vname, (cols, ng) in variants.items():
    txt = w[cols].fillna("").astype(str).agg(" ".join, axis=1)
    for a, b, label in folds:
        trm = (w["Open_Time"] < a).to_numpy()
        tem = ((w["Open_Time"] >= a) & (w["Open_Time"] < b)).to_numpy()
        m = token_model(ng).fit(txt[trm], w.loc[trm, "rfc"])
        p = m.predict_proba(txt[tem])[:, 1]
        y = w.loc[tem, "rfc"].to_numpy()
        rows.append({"Variant": vname, "Period": label, "Linked in top 50": int(y[np.argsort(-p)[:50]].sum())})
t16 = pd.DataFrame(rows).pivot(index="Variant", columns="Period", values="Linked in top 50")[periods]
t16["Total"] = t16.sum(axis=1)
print("Table 20. Linked tickets in the top 50 when one ingredient is changed")
display(t16)

Table 20. Linked tickets in the top 50 when one ingredient is changed


Period,Dec 2013,Jan 2014,Feb to 16 Mar 2014,Total
Variant,,,,
All fields (chosen),10,8,14,32
All fields + pairs of neighbouring fields,9,7,17,33
Without CI_Name,5,1,6,12
Without WBS,7,8,12,27


<details>
<summary><b>Code notes, line by line</b></summary>

- `variants = {...}` : The asset-token model with one ingredient changed: adding neighbour pairs (bigrams), dropping CI_Name, dropping WBS.
- `txt = w[cols].fillna("").astype(str).agg(" ".join, axis=1)` : Builds the text for that variant.
- `trm, tem = boolean masks` : Training and test rows of each fold.
- `m = token_model(ng).fit(txt[trm], ...)` : Fit and score each variant on each fold.
- `t16 = pivot(...); t16["Total"]` : Table 20: hits in the top 50 per variant.

</details>

**Result.** Dropping CI_Name cuts the top-50 total from 32 to 12; dropping WBS (27) or adding pairs of neighbouring fields (33) changes little (Table 20): the signal is mainly the individual asset.

## 8. Retrospective linking: are recent labels incomplete?

In ITIL practice a ticket can be linked to a change after it closes, for example during a problem review. If ABC Tech does this, tickets closed shortly before the extract have had less time to be linked, and their linked rate would drop. The five largest changes are left out so their bursts do not hide the pattern.

### 8a. Linked rate by time between closure and extract

In [14]:
extract_date = df["Close_Time"].max()
chk = w[~w["in_top5"]].copy()                                  # leave out the five storm changes
chk["Weeks before extract"] = ((extract_date - chk["Close_Time"]).dt.days // 7).clip(upper=12)
by_age = chk.groupby("Weeks before extract")["rfc"].agg(tickets="size", linked="sum")
by_age["Linked %"] = (by_age["linked"] / by_age["tickets"] * 100).round(2)
by_age = by_age.sort_index(ascending=False)
by_age.index = [f"{i}+ weeks" if i == 12 else f"{i} weeks" for i in by_age.index]
print(f"Table 21. Linked rate by how long before the extract ({extract_date.date()}) the ticket closed "
      "(five largest changes excluded)")
display(by_age)

Table 21. Linked rate by how long before the extract (2014-03-31) the ticket closed (five largest changes excluded)


,tickets,linked,Linked %
12+ weeks,23375,172,0.74
11 weeks,1822,4,0.22
10 weeks,1688,9,0.53
9 weeks,2025,8,0.40
8 weeks,2039,18,0.88
7 weeks,2088,12,0.57
6 weeks,1783,16,0.90
5 weeks,1763,8,0.45
4 weeks,1698,7,0.41
3 weeks,1646,12,0.73


<details>
<summary><b>Code notes, line by line</b></summary>

- `extract_date = df["Close_Time"].max()` : Last close date in the extract (taken as the extract date).
- `chk = w[~w["in_top5"]].copy()` : Leaves out the five storm changes.
- `((extract_date - Close_Time).dt.days // 7).clip(upper=12)` : Whole weeks between closure and the extract; 12 or more pooled.
- `groupby(...)["rfc"].agg(tickets="size", linked="sum"); Linked %` : Table 21: linked rate by age. A low rate in the most recent weeks would mean links are added late.
- `by_age.index = ["12+ weeks" ...]` : Readable row labels.

</details>

**Result.** No sign of retrospective linking (Table 21): tickets closed in the last two weeks before the extract (rows "0 weeks" and "1 weeks") are linked at 0.97% (6 of 620), in line with 0.59% for 2 to 11 weeks and 0.74% for older tickets (five largest changes excluded). The lower March rate (Table 1) is therefore not explained by late linking. The check cannot rule out links added months after closure.

## 9. Inferred CI relationships

Can related assets be inferred without a CMDB? Two methods, built on training months only: CIs linked to the same change, and CIs with tickets in the same hour more often than chance (chance weighted by busy hours; thresholds lift ≥ 3 and 5+ shared hours, set before running). No CI names are shown.

In [15]:
# Built from training months only
g = w[w["Open_Time"] < TEST_START].copy()
g["ci"] = g["CI_Name"].astype(str)

# 9a-1 CIs on the same change
cis_per_change = g[g["rfc"] == 1].groupby(g["Related_Change"].astype(str))["ci"].apply(lambda s: sorted(set(s)))
change_pairs = pd.DataFrame([p for cis in cis_per_change for p in combinations(cis, 2)],
                            columns=["a", "b"]).drop_duplicates()

# 9a-2 CIs with tickets in the same hour more often than chance (CIs with 30+ tickets)
counts = g["ci"].value_counts()
keep = pd.Index(sorted(counts[counts >= 30].index))
h = g.loc[g["ci"].isin(keep), ["Open_Time", "ci"]].assign(slot=lambda d: d["Open_Time"].dt.floor("h"))
h = h[["slot", "ci"]].drop_duplicates()
slots = pd.Index(sorted(h["slot"].unique()))
X = sparse.csr_matrix((np.ones(len(h)), (slots.get_indexer(h["slot"]), keep.get_indexer(h["ci"]))),
                      shape=(len(slots), len(keep)))
n = np.asarray(X.sum(axis=0)).ravel()                  # hours each CI had a ticket
a = np.asarray(X.sum(axis=1)).ravel()                  # active CIs in each hour
eff_hours = a.sum() ** 2 / (a ** 2).sum()              # effective number of hours, busy hours count more
co = (X.T @ X).tocoo()
m = co.row < co.col
pairs = pd.DataFrame({"a": keep[co.row[m]], "b": keep[co.col[m]], "together": co.data[m]})
pairs["lift"] = pairs["together"] / (n[co.row[m]] * n[co.col[m]] / eff_hours)
hour_pairs = pairs[(pairs["together"] >= 5) & (pairs["lift"] >= 3)]
print(f"effective hours {eff_hours:.0f} | same-hour pairs flagged: {len(hour_pairs)} "
      f"of {len(keep) * (len(keep) - 1) // 2} possible")

# 9a summary
main = g.groupby("ci")[["CI_Cat", "WBS"]].agg(lambda s: s.value_counts().index[0] if s.notna().any() else "unknown")
def share_same(p, col):
    return round((main.loc[p["a"], col].to_numpy() == main.loc[p["b"], col].to_numpy()).mean() * 100, 1) if len(p) else np.nan
both = change_pairs.merge(hour_pairs[["a", "b"]]).shape[0]
t9a = pd.DataFrame({
    "Pairs": [len(change_pairs), len(hour_pairs)],
    "CIs involved": [pd.unique(change_pairs[["a", "b"]].values.ravel()).size,
                     pd.unique(hour_pairs[["a", "b"]].values.ravel()).size],
    "Same CI_Cat %": [share_same(change_pairs, "CI_Cat"), share_same(hour_pairs, "CI_Cat")],
    "Same WBS %": [share_same(change_pairs, "WBS"), share_same(hour_pairs, "WBS")],
}, index=["Same change", "Same hour (lift >= 3, 5+ hours)"])
print(f"Table 22. Inferred CI relationships, training months | pairs found by both methods: {both}")
display(t9a)

# 9b Does the map help? Test-period linked rate for CIs related to a CI that had a linked ticket in training
edges = pd.concat([change_pairs, hour_pairs[["a", "b"]]]).drop_duplicates()
nbrs = pd.concat([edges.rename(columns={"a": "ci", "b": "nb"}), edges.rename(columns={"b": "ci", "a": "nb"})])
hot = set(g.loc[g["rfc"] == 1, "ci"])
near_hot = set(nbrs.loc[nbrs["nb"].isin(hot), "ci"]) - hot
t = te.assign(ci=te["CI_Name"].astype(str))
t["Group"] = np.select([t["ci"].isin(hot), t["ci"].isin(near_hot)],
                       ["CI itself had a linked ticket in training", "Related to such a CI (not itself)"], "Neither")
t9b = t.groupby("Group").agg(Tickets=("rfc", "size"), Linked=("rfc", "sum"), Linked_without_top5=("rfc_ex", "sum"))
t9b["Linked %"] = (t9b["Linked"] / t9b["Tickets"] * 100).round(2)
t9b["Linked % without top 5"] = (t9b["Linked_without_top5"] / t9b["Tickets"] * 100).round(2)
print("Table 23. Test period (1 Feb to 16 Mar): linked rate by relationship to CIs with past links")
display(t9b)

effective hours 807 | same-hour pairs flagged: 13 of 10011 possible
Table 22. Inferred CI relationships, training months | pairs found by both methods: 0


,Pairs,CIs involved,Same CI_Cat %,Same WBS %
Same change,65,33,64.6,30.8
"Same hour (lift >= 3, 5+ hours)",13,21,53.8,15.4


Table 23. Test period (1 Feb to 16 Mar): linked rate by relationship to CIs with past links


,Tickets,Linked,Linked_without_top5,Linked %,Linked % without top 5
Group,,,,,
CI itself had a linked ticket in training,5182,66,39,1.27,0.75
Neither,5288,73,17,1.38,0.32
Related to such a CI (not itself),87,0,0,0.00,0.00


<details>
<summary><b>Code notes, line by line</b></summary>

- `g = w[w["Open_Time"] < TEST_START].copy()` : Training months only, so the test check in 9b is fair.
- `cis_per_change = ...groupby(change)["ci"].apply(lambda s: sorted(set(s)))` : The set of CIs on each change.
- `change_pairs = pd.DataFrame([p for cis in ... for p in combinations(cis, 2)])` : Method 1: every pair of CIs on the same change.
- `keep = CIs with 30+ tickets` : Method 2 uses only CIs with enough tickets.
- `h = ...assign(slot=Open_Time.dt.floor("h"))...drop_duplicates()` : Which CIs had a ticket in each clock hour.
- `X = sparse.csr_matrix(...)` : Hours x CIs table of 0/1.
- `n = X.sum(axis=0); a = X.sum(axis=1)` : Active hours per CI, and active CIs per hour.
- `eff_hours = a.sum() ** 2 / (a ** 2).sum()` : Effective number of hours: busy hours count more, so chance co-occurrence is not underestimated.
- `co = (X.T @ X).tocoo(); m = co.row < co.col` : Hours shared by each CI pair (each pair once).
- `pairs["lift"] = together / (n_a * n_b / eff_hours)` : Observed shared hours divided by the number expected by chance.
- `hour_pairs = pairs[(together >= 5) & (lift >= 3)]` : Pairs sharing 5+ hours at 3 times chance or more.
- `main = ...; def share_same(p, col): ...` : Each CI's usual CI_Cat and WBS, and the % of pairs that share them.
- `t9a = ...` : Table 22: pairs and CIs per method, and the same-type / same-WBS shares.
- `edges / nbrs / hot / near_hot` : The relationship map; CIs with a linked ticket in training ('hot') and their related CIs.
- `t["Group"] = np.select([...], [...], "Neither")` : Assigns each test ticket to one of three groups.
- `t9b = ...` : Table 23: test-period linked rate per group, with and without the storms.

</details>

**Result.** CIs linked to the same change give 65 pairs (33 CIs; 64.6% share an asset type and 30.8% a WBS); same-hour tickets give 13 pairs (21 CIs); the two methods share no pairs (Table 22). In the test period, the 87 tickets on CIs related to a CI with past links had no linked change, while CIs with their own past links were linked at 0.75% against 0.32% for the rest (five largest changes excluded; Table 23). The signal sits in the asset itself, which the asset-token watchlist already uses (Table 20); inferred relationships do not extend it. A confirmed dependency map would need the CMDB.

## 10. Likely failure type at creation (Closure_Code)
Closure_Code records what the cause turned out to be. Five codes name a failure type: Software, Hardware, Data, User error, and No error (works as designed). A model that predicts the code from creation-time fields (Set B plus CI_Name, with CI_Names seen fewer than 5 times pooled as in 7c: failures are asset-specific, and CI_Name raised test macro-F1 from about 0.51 to 0.58 in a check) suggests the likely cause when a ticket opens, for routing and for spotting assets with repeated hardware or data failures. Closure_Code is only the target: it is recorded at closure and is never a feature. Same window and split as the other sections; tuned by macro-F1 with time-ordered CV and balanced weights; model choice uses the Track 1 rule (best CV macro-F1, logistic regression kept if within 0.01).

In [16]:
FAILURE_TYPES = ["Software", "Hardware", "Data", "User error", "No error - works as designed"]
ft = w[w["Closure_Code"].isin(FAILURE_TYPES)].sort_values("Open_Time")
tr_f, te_f = ft[ft["Open_Time"] < TEST_START], ft[ft["Open_Time"] >= TEST_START]
to_code = {c: k for k, c in enumerate(FAILURE_TYPES)}
Xtr_f, Xte_f = tr_f[cols_ci].astype(str), te_f[cols_ci].astype(str)        # Set B + CI_Name, as in 7c
ytr_f, yte_f = tr_f["Closure_Code"].map(to_code), te_f["Closure_Code"].map(to_code).to_numpy()
print(f"Table 24. Tickets per failure type ({len(ft):,} of {len(w):,} window tickets; "
      "Other, Unknown and the rarer codes are left out)")
display(pd.DataFrame({"Train": tr_f["Closure_Code"].value_counts(), "Test": te_f["Closure_Code"].value_counts()}
                     ).reindex(FAILURE_TYPES))

fail_models = {
    "Logistic Regression": (LogisticRegression(max_iter=3000), {"clf__C": [0.01, 0.1, 1, 10]}),
    "Random Forest": (RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
        {"clf__max_depth": [10, 20], "clf__min_samples_leaf": [5, 20]}),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=42),
        {"clf__learning_rate": [0.05, 0.1], "clf__max_depth": [2, 3]}),
    "XGBoost": (XGBClassifier(objective="multi:softprob", n_estimators=200, eval_metric="mlogloss",
                              random_state=42, n_jobs=-1),
        {"clf__max_depth": [3, 5], "clf__learning_rate": [0.05, 0.1]}),
}
wts_f = compute_sample_weight("balanced", ytr_f)
fitted_f = {}
for name, (clf, grid) in fail_models.items():
    gs = GridSearchCV(onehot_ci(clf), grid, cv=TimeSeriesSplit(n_splits=3), scoring="f1_macro", n_jobs=1)
    gs.fit(Xtr_f, ytr_f, clf__sample_weight=wts_f)
    fitted_f[name] = gs

def ft_row(name, pred, cv=np.nan):
    return {"Model": name, "CV macro-F1 (train)": cv,
            "Test accuracy": accuracy_score(yte_f, pred),
            "Balanced accuracy": balanced_accuracy_score(yte_f, pred),
            "Macro-F1": f1_score(yte_f, pred, average="macro"),
            "Weighted F1": f1_score(yte_f, pred, average="weighted"),
            "Hardware recall": recall_score(yte_f, pred, labels=[1], average="macro", zero_division=0)}

# Baselines: always the most common type; the most common type of the ticket's CI_Subcat x Category
majority_f = int(ytr_f.mode().iloc[0])
lookup_f = tr_f.assign(y=ytr_f).groupby(["CI_Subcat", "Category"])["y"].agg(lambda s: s.mode().iloc[0]).rename("lookup")
pred_lookup_f = (te_f[["CI_Subcat", "Category"]]
                 .merge(lookup_f, left_on=["CI_Subcat", "Category"], right_index=True, how="left")["lookup"]
                 .fillna(majority_f).astype(int).to_numpy())

rows = [ft_row(n, g_.best_estimator_.predict(Xte_f), g_.best_score_) for n, g_ in fitted_f.items()]
rows += [ft_row("Baseline: always the most common type", np.full(len(yte_f), majority_f)),
         ft_row("Baseline: most common type per CI_Subcat x Category", pred_lookup_f)]
print(f"Table 25. Failure type at creation, test period ({len(yte_f):,} tickets)")
display(pd.DataFrame(rows).set_index("Model").round(3))

cv_f = {k: g_.best_score_ for k, g_ in fitted_f.items()}
top_f = max(cv_f, key=cv_f.get)
ft_name = "Logistic Regression" if cv_f[top_f] - cv_f["Logistic Regression"] < 0.01 else top_f
pred_f = fitted_f[ft_name].best_estimator_.predict(Xte_f)
print(f"Top by CV macro-F1: {top_f} ({cv_f[top_f]:.3f}) | chosen: {ft_name} ({cv_f[ft_name]:.3f})")
print(classification_report(yte_f, pred_f, labels=list(range(5)), target_names=FAILURE_TYPES, digits=3, zero_division=0))
print("Table 26. Actual against suggested failure type, chosen model")
display(pd.DataFrame(confusion_matrix(yte_f, pred_f, labels=list(range(5))),
                     index=[f"Actual: {t}" for t in FAILURE_TYPES], columns=[f"Suggested: {t}" for t in FAILURE_TYPES]))

Table 24. Tickets per failure type (23,019 of 42,304 window tickets; Other, Unknown and the rarer codes are left out)


,Train,Test
Closure_Code,,
Software,8884,2896
Hardware,2102,649
Data,1479,555
User error,2250,964
No error - works as designed,2443,797


Table 25. Failure type at creation, test period (5,861 tickets)


,CV macro-F1 (train),Test accuracy,Balanced accuracy,Macro-F1,Weighted F1,Hardware recall
Model,,,,,,
Logistic Regression,0.545,0.587,0.635,0.582,0.595,0.901
Random Forest,0.549,0.600,0.608,0.574,0.603,0.898
Gradient Boosting,0.560,0.603,0.640,0.597,0.611,0.892
XGBoost,0.556,0.626,0.641,0.606,0.631,0.894
Baseline: always the most common type,NaN,0.494,0.200,0.132,0.327,0.000
Baseline: most common type per CI_Subcat x Category,NaN,0.593,0.402,0.392,0.508,0.726


Top by CV macro-F1: Gradient Boosting (0.560) | chosen: Gradient Boosting (0.560)
                              precision    recall  f1-score   support

                    Software      0.779     0.547     0.643      2896
                    Hardware      0.719     0.892     0.796       649
                        Data      0.491     0.564     0.525       555
                  User error      0.536     0.632     0.580       964
No error - works as designed      0.360     0.563     0.439       797

                    accuracy                          0.603      5861
                   macro avg      0.577     0.640     0.597      5861
                weighted avg      0.648     0.603     0.611      5861

Table 26. Actual against suggested failure type, chosen model


,Suggested: Software,Suggested: Hardware,Suggested: Data,Suggested: User error,Suggested: No error - works as designed
Actual: Software,1584,143,170,399,600
Actual: Hardware,51,579,8,4,7
Actual: Data,95,23,313,39,85
Actual: User error,128,21,99,609,107
Actual: No error - works as designed,176,39,48,85,449


<details>
<summary><b>Code notes, line by line</b></summary>

- `FAILURE_TYPES = [...]` : The five Closure_Code values that name a failure type.
- `ft = w[w["Closure_Code"].isin(FAILURE_TYPES)].sort_values("Open_Time")` : Tickets with one of them, in time order.
- `tr_f, te_f` : Same split dates.
- `to_code = {c: k ...}` : Maps each type to 0-4 (XGBoost needs numbers).
- `Xtr_f, Xte_f = ...[cols_ci].astype(str)` : Set B + CI_Name, as text categories.
- `display(pd.DataFrame({"Train": ..., "Test": ...}))` : Table 24: tickets per type.
- `fail_models = {...}` : Four multi-class models.
- `wts_f = compute_sample_weight("balanced", ytr_f)` : Equal total weight per type.
- `GridSearchCV(onehot_ci(clf), ..., scoring="f1_macro")` : Tuned by macro-F1 with time-ordered CV.
- `def ft_row(...)` : Accuracy, balanced accuracy, macro-F1, weighted F1 and Hardware recall.
- `majority_f; lookup_f` : Baselines: always the most common type; most common type per CI_Subcat x Category.
- `display(pd.DataFrame(rows)...)` : Table 25.
- `ft_name = "Logistic Regression" if ... < 0.01 else top_f` : The project's tie-break rule.
- `classification_report(...); confusion_matrix(...)` : Per-type scores and Table 26.

</details>

**Result.** On the test period the four tuned models reach macro-F1 0.574 to 0.606 and balanced accuracy 0.608 to 0.641, against 0.392 and 0.402 for the CI_Subcat x Category lookup and 0.132 and 0.200 for always suggesting Software (Table 25). Accuracy (0.59 to 0.63) is close to the lookup’s 0.593, because the lookup mostly suggests Software, the largest type; the models are much better on the smaller types. Gradient boosting has the best CV macro-F1 (0.560) and is chosen; XGBoost is slightly ahead on the test period (macro-F1 0.606 against 0.597), a difference within noise. Hardware is suggested most reliably (recall 0.89, precision 0.72); Data and “No error” are the weakest (F1 0.53 and 0.44), and a fifth of Software tickets are suggested as “No error” (Table 26). Closure_Code is filled in by the engineer at closure, so the labels record their judgement, and 46% of window tickets (Other, Unknown and the rarer codes) fall outside the five types.

### 10b. Using the probabilities: confident suggestions and a hardware ranking
The model gives a probability for each failure type. Table 27 suggests a type only when the top probability passes a threshold; Table 28 ranks tickets by their probability of being Hardware.

In [17]:
# Using the probabilities: suggest only when confident, and rank tickets by P(Hardware)
model_f = fitted_f[ft_name].best_estimator_
proba_f = model_f.predict_proba(Xte_f)
top_p, top_k = proba_f.max(axis=1), proba_f.argmax(axis=1)

rows = []
for t in [0.0, 0.4, 0.5, 0.6, 0.7, 0.8]:
    m = top_p >= t
    rows.append({"Minimum top probability": "none (every ticket)" if t == 0 else f"{t:.1f}",
                 "Tickets with a suggestion": int(m.sum()),
                 "Coverage (%)": round(m.mean() * 100, 1),
                 "Suggestion correct (%)": round((top_k[m] == yte_f[m]).mean() * 100, 1) if m.any() else np.nan})
print(f"Table 27. Suggest a failure type only when the model is confident ({ft_name}, test period)")
display(pd.DataFrame(rows).set_index("Minimum top probability"))

hw = FAILURE_TYPES.index("Hardware")
order = np.argsort(-proba_f[:, hw])
base = (yte_f == hw).mean()
rows = []
for share in [0.05, 0.10, 0.20]:
    k = int(round(share * len(order)))
    hit = (yte_f[order[:k]] == hw).mean()
    rows.append({"Tickets flagged (%)": int(share * 100), "Tickets flagged": k,
                 "Actually Hardware (%)": round(hit * 100, 1), "Lift over base rate": round(hit / base, 1),
                 "Share of all Hardware tickets caught (%)": round((yte_f[order[:k]] == hw).sum() / (yte_f == hw).sum() * 100, 1)})
print(f"Table 28. Tickets ranked by P(Hardware) (test base rate {base * 100:.1f}%)")
display(pd.DataFrame(rows).set_index("Tickets flagged (%)"))

Table 27. Suggest a failure type only when the model is confident (Gradient Boosting, test period)


,Tickets with a suggestion,Coverage (%),Suggestion correct (%)
Minimum top probability,,,
none (every ticket),5861,100.0,60.3
0.4,3290,56.1,74.3
0.5,2686,45.8,78.2
0.6,2123,36.2,79.8
0.7,1366,23.3,81.3
0.8,837,14.3,86.4


Table 28. Tickets ranked by P(Hardware) (test base rate 11.1%)


,Tickets flagged,Actually Hardware (%),Lift over base rate,Share of all Hardware tickets caught (%)
Tickets flagged (%),,,,
5,293,83.6,7.6,37.8
10,586,78.2,7.1,70.6
20,1172,52.9,4.8,95.5


<details>
<summary><b>Code notes, line by line</b></summary>

- `model_f = fitted_f[ft_name].best_estimator_; proba_f = model_f.predict_proba(Xte_f)` : Probability of each of the five types for every test ticket (rows sum to 1).
- `top_p, top_k = proba_f.max(axis=1), proba_f.argmax(axis=1)` : The highest probability and the type it belongs to.
- `for t in [0.0, 0.4, ..., 0.8]: m = top_p >= t` : Suggest a type only when the top probability reaches t.
- `"Coverage (%)", "Suggestion correct (%)"` : Table 27: share of tickets that get a suggestion, and how often it is right.
- `hw = FAILURE_TYPES.index("Hardware"); order = np.argsort(-proba_f[:, hw])` : Tickets ranked by P(Hardware).
- `base = (yte_f == hw).mean()` : Hardware base rate in the test period.
- `for share in [0.05, 0.10, 0.20]: ...` : Table 28: % hardware among the top-ranked tickets, lift, and share of all hardware tickets caught.

</details>

**Result.** Suggesting a type for every ticket is right 60% of the time. Requiring a top probability of at least 0.5 gives suggestions for 46% of tickets, right 78% of the time; at 0.7, 23% of tickets, right 81% (Table 27). The rest get no suggestion. Ranked by P(Hardware), the top 10% of tickets are 78% Hardware (7 times the 11% base rate) and contain 71% of all Hardware tickets (Table 28), so likely hardware faults can be routed to hardware support at creation. The probabilities come from a model trained with balanced weights, so they are relative scores rather than calibrated chances; thresholds should be set on validation data before use.

## 11. Findings and decision

1. **Target.** Has_Related_Change: 516 of 42,304 tickets (1.22%) in the modelling window (Table 1), belonging to 205 distinct changes. The median change links 1 ticket; the largest links 110 (Table 6). The 5 largest changes cover 232 tickets (45%): short bursts on 1 or 2 WBS codes (4-8 Nov, 17-27 Feb, 2 Dec, 14-18 Nov), plus one slow change spread over five months (Tables 7 and 10).
2. **Bursts do not predict a link.** The linked rate does not rise with the number of tickets opened on the same WBS or CI in the previous 24 hours; without the 5 largest changes it falls (Table 11: 0.25% and 0.35% for the busiest WBS groups, against 0.67% for all tickets outside the five largest changes). Tickets with no recent ticket on the same WBS have the highest rate (1.89%).
3. **Test period is dominated by one event.** Train: 377 positives; test (1 Feb to 16 Mar): 139, of which 78 belong to one change (Tables 7 and 10). Only 6 test positives belong to a change that also has training tickets (Table 7).
4. **All five algorithms, tuned with time-ordered CV, on asset and time fields.** All tickets (Table 13): average precision 1.0 to 1.4 times the base rate, ROC-AUC 0.38 to 0.45 (worse than random). Without the 5 largest changes (Table 14; 228 train, 56 test positives): average precision 4.3 to 6.4 times base (0.023 to 0.034 absolute), ROC-AUC 0.65 to 0.75, precision 1.2% to 2.8%. Accuracy stays below the always-"no change" figure.
5. **Caveats.** The 5 largest changes were chosen using all months, test included, so the second result is a post-hoc view. Only 56 positives from about 45 changes remain. Has_Related_Change is probably set after the ticket is created, so the lower March rate (0.4%, Table 1) was checked for late linking in section 8; none was found.
6. **Retrospective linking (section 8).** No sign of it: the linked rate for tickets closed shortly before the extract is in line with older tickets (Table 21), so Has_Related_Change is used as recorded and no label cutoff is applied.
7. **Inferred CI relationships (section 9).** Built from shared changes and same-hour tickets; the two methods share no pairs (Table 22), and related CIs were not linked more often in the test period (Table 23). Not used.
8. **Likely failure type (section 10).** From creation-time fields plus CI_Name, five failure types (Software, Hardware, Data, User error, No error) are suggested with macro-F1 0.57 to 0.61 (gradient boosting chosen) against 0.39 for a lookup (Table 25); Hardware is the most reliable (recall about 0.9, Table 26). It covers the 54% of window tickets that carry one of the five codes (Table 24). Used with a confidence threshold, the suggestion is right 78% of the time for the 46% of tickets above 0.5 (Table 27).

### Decision
Most linked changes cannot be predicted from creation-time fields: 45% come from five incident storms (Table 6), and five tuned classifiers on one-hot asset and time fields are at or below chance on all tickets (Table 13). K-means clustering (k = 5 to 40) as a risk score was also no better than chance.

A narrow watchlist signal exists: LightGBM on the ticket's asset codes (CI_Name, CI_Cat, CI_Subcat, WBS, Category; every code kept, including rare CIs; TF-IDF weighted). Trained on earlier months only, its 50 highest-scored tickets contained 10 (Dec), 8 (Jan) and 14 (1 Feb to 16 Mar) linked tickets, against 0.6, 0.3 and 0.7 expected by chance (Table 18); mean average precision 0.086 against about 0.011 (Table 19). In the test period the 21 linked tickets in its top 100 came from 17 different changes, all on CIs already seen in training (Table 17): the model flags recurring assets, not storms.

The signal comes mainly from CI_Name: without it the top-50 total falls from 32 to 12; adding pairs of fields (33) or removing WBS (27) changes little (Table 20). One-hot encoding that pools CIs with fewer than 5 training rows keeps less of it (15 with XGBoost, 4 with Logistic Regression; Table 18).

Use: a short ranked list for change or problem management, covering only a small share of linked tickets.

A second, broader deliverable is the failure-type suggestion (section 10): when a ticket opens, its likely cause, with Hardware the most reliable type (Tables 25 to 28). It supports routing (to hardware or software support) and, counted per asset, points to CIs with repeated hardware or data failures. Closure_Code stands in for a failure label, so it describes recorded causes, not confirmed failures or misconfigurations. TF-IDF, a text-vectorisation technique from NLP, is used here only as an encoder for asset codes; the extract contains no free text.